In [1]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from ranking_methods import rank_accuracy
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau

                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [5]:
best_feat = 'ibi_median'
data_folder = "./data/dados_2026_06_08"
dataFiles = glob.glob(f'{data_folder}/*.zip')
output_folder = './results/predict_2026_06_08'
os.makedirs(output_folder, exist_ok=True)
print(dataFiles)

['./data/dados_2026_06_08/2026-05-14 17.07.20.zip', './data/dados_2026_06_08/2026-05-18 12.26.42.zip', './data/dados_2026_06_08/2026-05-18 11.29.53.zip', './data/dados_2026_06_08/2026-05-15 16.33.28.zip', './data/dados_2026_06_08/2026-05-19 12.10.38.zip', './data/dados_2026_06_08/2026-05-12 11.46.29.zip', './data/dados_2026_06_08/2026-05-07 14.29.23.zip', './data/dados_2026_06_08/2026-05-08 14.28.22.zip', './data/dados_2026_06_08/2026-05-11 11.49.16.zip', './data/dados_2026_06_08/2026-05-22 18.27.08.zip', './data/dados_2026_06_08/2026-05-20 15.08.27.zip']


In [6]:
root_folder = f"{data_folder}"    

for file in dataFiles:
    zip_folder = file.split("/")[-1]
    sub_folder = zip_folder.split(".")[0].replace(" ", "_")
    os.makedirs(os.path.join(root_folder, sub_folder), exist_ok=True)
    a = chr.intellicage_unwrapper([file], sub_folder, sampling_interval = '30T')



File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_1.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_10.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_11.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_12.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_2.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_3.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_4.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_5.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_6.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_7.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_8.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_9.txt
File saved in ./data/dados_2026_06_08/2026-05-18_12/animal_1.txt
File saved in ./data/dados_2026_06_08/2026-05-18_12/animal_10.txt
File saved in ./data/dados_2026_06_08/2026-05-18_12/animal_11.txt
File saved in ./data

In [11]:
individual_files = glob.glob(root_folder + "/**/*.txt", recursive=True)
individual_files = [f for f in individual_files if "animal_" in f]
apply_filtering = True
animals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)

Animal 1: 10
Animal 2: 11
Animal 3: 10
Animal 4: 11
Animal 5: 10
Animal 6: 11
Animal 7: 10
Animal 8: 11
Animal 9: 10
Animal 10: 10
Animal 11: 11
Animal 12: 10
Animal2 1
savgol True
Animal2 2
savgol True
Animal2 3
savgol True
Animal2 4
savgol True
Animal2 5
savgol True
Animal2 6
savgol True
Animal2 7
savgol True
Animal2 8
savgol True
Animal2 9
savgol True
Animal2 10
savgol True
Animal2 11
savgol True
Animal2 12
savgol True
Animal animal_1 Days found: 22
Animal animal_2 Days found: 22
Animal animal_3 Days found: 22
Animal animal_4 Days found: 22
Animal animal_5 Days found: 22
Animal animal_6 Days found: 22
Animal animal_7 Days found: 22
Animal animal_8 Days found: 22
Animal animal_9 Days found: 22
Animal animal_10 Days found: 22
Animal animal_11 Days found: 22
Animal animal_12 Days found: 22


In [12]:
output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

X_scaled =  get_data_scaled(all_features, feature_cols)
all_features.head()

Saving features on ./results/predict_2026_06_08/basic_features.csv
Saving temporal features on ./results/predict_2026_06_08/temporal_features.csv
Saving all features on ./results/predict_2026_06_08/all_features.csv


,animal,total_activity,mean_activity,std_activity,max_activity,min_activity,cv_activity,median_activity,max_median_ratio,activity_per_hour,...,night_ibi_cv,night_bout_len_mean,night_bout_len_cv,night_transitions_per_hour,night_onset_latency_h,night_first_2h_frac,night_gini,night_peak_hour,night_activity_per_bout,night_day_intensity_ratio
animal_1,animal_1,2180.0,1.081886,1.849510,14.057143,-2.000000,1.709524,0.342857,4.100000e+01,90.833333,...,1.897390,0.605769,0.767178,0.826216,4.00,0.000000,0.000000,0.0,-1.699829,-0.083451
animal_2,animal_2,1773.0,0.879901,1.425818,13.028571,-1.800000,1.620431,0.142857,9.120000e+01,73.875000,...,2.070792,0.583333,0.685119,0.857994,4.25,-0.040874,4.792238,0.0,0.850136,0.085291
animal_3,animal_3,1252.0,0.621340,1.128312,7.200000,-1.457143,1.815934,0.000000,7.200000e+09,52.166667,...,2.203825,0.572727,0.709597,0.873883,4.25,-0.024671,3.194498,1.0,1.337723,0.185508
animal_4,animal_4,2381.0,1.181638,1.850439,9.600000,-1.457143,1.565996,0.342857,2.800000e+01,99.208333,...,2.158107,0.594340,0.734159,0.842105,4.25,-0.021011,2.791358,1.0,1.729549,0.117113
animal_5,animal_5,2620.0,1.300248,1.934039,21.600000,-2.057143,1.487439,0.485714,4.447059e+01,109.166667,...,2.298396,0.543103,0.648952,0.921549,10.25,-0.023178,2.998585,1.0,1.392039,0.102516


In [13]:
feature_rhos_path = "./data/feature_rhos_new.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos_new = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos_new.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [ ]:


best_feat = 'cosinor_amplitude'
combos = [['power_24h', 'high_activity_frac', 'activity_per_bout'], ['power_24h', 'short_gap_frac', 'night_ibi_cv']]



result = []
cont = 0
for named_combo in combos:

    feature_rhos, proxies_raw = build_all_proxies(
        all_features, feature_cols, X_scaled, None,
        k=3, best_feat_idx=best_feat,
        named_combo=named_combo,
        feature_rhos=feature_rhos_new
    )
    for k, v in proxies_raw.items():
        print(f"{k}: {v}")



    best_feature_key = f'Best feature ({best_feat})'

    scores = proxies_raw[best_feature_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)
 
    if cont == 0:
        result.append({"name": best_feat, "pred": pred_rank})

    output_best_feature = f'{data_folder}/pred_{best_feat}.csv'
    #print(f"Saving output best feat {output_best_feature}")

    best_combo_key = f'Best combo ({ " + ".join(named_combo) })'
    #print(best_combo_key)
    scores = proxies_raw[best_combo_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)


    combo_name = "_".join(named_combo)
    result.append({"name": combo_name, "pred": pred_rank})
    cont += 1


df = pd.DataFrame(result)
df.to_csv(f"{root_folder}/pred_2026_06_08.csv", index=False)

Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.31492114 0.23325785 0.41784596 0.45126302 0.40910995 0.3495697
 0.29124483 0.25314785 0.35359512 0.26930195 0.34955908 0.25913889]
Best combo (power_24h + high_activity_frac + activity_per_bout): [-1.97743454 -0.64009066  4.30367554  1.54514224 -0.70545626  1.45826443
 -0.99261139  0.65296058 -0.25201251 -2.30812169 -1.21073732  0.12642158]
Combo sign-aligned mean (power_24h + high_activity_frac + activity_per_bout): [-0.65914485 -0.21336355  1.43455851  0.51504741 -0.23515209  0.48608814
 -0.33087046  0.21765353 -0.08400417 -0.7693739  -0.40357911  0.04214053]
Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.31492114 0.23325785 0.41784596 0.45126302 0.40910995 0.3495697
 0.29124483 0.25314785 0.35359512 0.26930195 0.34955908 0.25913889]
Best combo (power_24h + short_gap_frac + night_ibi_cv): [-0.78702875 -0.40065014  1

In [ ]:
import numpy as np
from scipy.stats import spearmanr

def rank_accuracy(actual_ranks, proxy_ranks):
    """Compare two integer rank arrays and return accuracy metrics.

    Parameters
    ----------
    actual_ranks : array-like of int
        Ground-truth rank for each animal (1 = most dominant).
    proxy_ranks  : array-like of int
        Predicted rank for each animal.

    Returns
    -------
    dict with keys:
        accuracy  – fraction of animals with exact rank match
        within_1  – fraction with |error| <= 1
        within_2  – fraction with |error| <= 2
        mae       – mean absolute error
        rho       – Spearman correlation
    """
    actual = np.asarray(actual_ranks, dtype=float)
    pred   = np.asarray(proxy_ranks,  dtype=float)
    abs_err = np.abs(pred - actual)
    return {
        'accuracy': float(np.mean(abs_err == 0)),
        'within_1': float(np.mean(abs_err <= 1)),
        'within_2': float(np.mean(abs_err <= 2)),
        'mae':      float(np.mean(abs_err)),
        'rho':      float(spearmanr(actual, pred).correlation),
    }




ranks_ground = [6,1,11,10,12,8,5,9,2,4,7,3]

for r in result:
    print(r['name'])
    print(rank_accuracy(r['pred'], ranks_ground))
    print()



cosinor_amplitude
{'accuracy': 0.6666666666666666, 'within_1': 0.6666666666666666, 'within_2': 0.8333333333333334, 'mae': 1.5, 'rho': 0.6293706293706295}

power_24h_high_activity_frac_activity_per_bout
{'accuracy': 0.08333333333333333, 'within_1': 0.3333333333333333, 'within_2': 0.4166666666666667, 'mae': 3.1666666666666665, 'rho': 0.39860139860139865}

power_24h_short_gap_frac_night_ibi_cv
{'accuracy': 0.16666666666666666, 'within_1': 0.3333333333333333, 'within_2': 0.5, 'mae': 2.6666666666666665, 'rho': 0.5594405594405596}

